In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# NVIDIA Ising Calibration Introductory Demo

$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

---

**What You Will Do:**
* Connect a Jupyter notebook to the hosted NVIDIA Ising Calibration API
* Learn how to structure a reliable zero-shot prompt
* Analyze either one calibration plot or several related plots using NVIDIA Ising Calibration
* Learn how to structure a reliable in-context learning prompt using an expert-labeled example
* Compare zero-shot and ICL analyses of the same Ramsey query containing a synthetic calibration error.

**Prerequisites:**
* Python and Jupyter notebook familiarity
* A Python environment with the `openai` package installed
* An NVIDIA API key stored in the `NVIDIA_API_KEY` environment variable
* Basic familiarity with superconducting-qubit calibration experiments
* An extracted `calibration_images` folder stored beside this notebook (see [README.md](README.md))
* If needed: [Environment and API-key setup guide](https://nvidia.github.io/cuda-q-academic/interactive_widgets/ising_api_key_setup.html)

**Key Terminology:**
* Zero-shot inference
* Experimental context
* Vision-language model (VLM)
* In-context learning (ICL)
* Labeled demonstration
* Query plot


In [ ]:
## Uncomment the line below and execute this cell to install the API client.

#!pip install openai -q

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Kernel → Restart** in Jupyter).

In [ ]:
# Python standard library
import base64
import os
from pathlib import Path

# API client and notebook display
from IPython.display import Markdown, display
from openai import OpenAI

# Part I — Zero-Shot Analysis

---
## 1. Connect to NVIDIA Ising Calibration

**Zero-shot inference** means asking the model to analyze a new plot without first giving it solved examples in the same conversation. NVIDIA Ising Calibration is a **vision-language model (VLM)**: it receives both text and one or more plot images.

This notebook calls NVIDIA's hosted API, so it does not require a local GPU or Docker. It does require the `NVIDIA_API_KEY` environment variable. Set the key before launching Jupyter so the selected notebook kernel inherits it.

The model identifier and recommended zero-shot defaults below match the current NVIDIA Ising Calibration 1.5 [hosted model card](https://build.nvidia.com/nvidia/ising-calibration-1.5-31b/modelcard).

In [ ]:
MODEL = "nvidia/ising-calibration-1.5-31b"
BASE_URL = "https://integrate.api.nvidia.com/v1"
CALIBRATION_IMAGE_DIR = Path("calibration_images")

if not CALIBRATION_IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "The calibration_images folder is missing. Extract "
        "images/calibration_images.zip beside this notebook "
        "(see README.md), then rerun this cell."
    )

api_key = os.environ.get("NVIDIA_API_KEY")
if not api_key:
    raise RuntimeError(
        "NVIDIA_API_KEY is not available. Set it in the terminal, restart "
        "Jupyter from that terminal, and select the kernel that inherits "
        "the updated environment."
    )

client = OpenAI(base_url=BASE_URL, api_key=api_key)
print(f"Ready to call {MODEL}")


---
## 2. Define reusable image helpers

The API accepts PNG and JPEG images as base64-encoded data URLs. These helpers look up simple filenames inside the `calibration_images` folder, validate every path before making a model request, and support either one filename or a list of filenames.

In [ ]:
def resolve_plot_paths(plots):
    """Return validated plot paths from one path or a list of paths."""
    if isinstance(plots, (str, Path)):
        plots = [plots]

    paths = []
    for plot in plots:
        path = Path(plot)
        if not path.is_absolute() and path.parent == Path("."):
            path = CALIBRATION_IMAGE_DIR / path
        paths.append(path)
    if not paths:
        raise ValueError("Provide at least one calibration plot.")

    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing plot file(s): "
            + ", ".join(missing)
            + ". Confirm that calibration_images is beside the notebook "
            + "and contains the expected files."
        )

    return paths


def image_part(path):
    """Convert one PNG or JPEG plot into an API image content item."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix not in {".png", ".jpg", ".jpeg"}:
        raise ValueError(f"{path.name} is not a PNG or JPEG image.")

    mime_type = "image/jpeg" if suffix in {".jpg", ".jpeg"} else "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")

    return {
        "type": "image_url",
        "image_url": {"url": f"data:{mime_type};base64,{encoded}"},
    }

---
## 3. Experimental context and analysis tasks

Good **experimental context** tells Ising what experiment produced the plot, what was swept, what was measured, the units, the objective, and any known limitations. It should not tell the model what conclusion to reach. Context is optional so that you can compare the model's response with and without it.

The **analysis task** below follows Ising Calibration's intended analysis categories: technical description, experimental conclusion, significance, fit reliability, parameter extraction, and success classification. 

In [ ]:
generic_analysis_task = """
Analyze the supplied calibration plot or plots using only the visible
evidence and any experimental context provided.

Return these sections:

1. TECHNICAL DESCRIPTION
   Describe the axes, plotted quantities, and important visible features.

2. EXPERIMENTAL CONCLUSION
   State what the measurement directly supports.

3. EXPERIMENTAL SIGNIFICANCE
   Explain what the result means for the calibration workflow.

4. FIT QUALITY ASSESSMENT
   Assess raw-data quality separately from the quality of any displayed fit.
   Do not assume printed fit values are trustworthy.

5. PARAMETER EXTRACTION
   Extract only parameters supported by the image and any provided context. Include units
   and image-supported precision. Return null for unsupported values.

6. CALIBRATION STATUS CLASSIFICATION
   Choose exactly one:
   EXPECTED_BEHAVIOR, SUBOPTIMAL_PARAMETERS, ANOMALOUS_BEHAVIOR,
   POSSIBLE_APPARATUS_ISSUE, or INSUFFICIENT_INFORMATION.

7. RECOMMENDED NEXT STEP
   Recommend one bounded, testable next calibration action.

For every conclusion, cite visible evidence. Clearly distinguish direct
observations from inferences. Do not invent missing settings, uncertainties,
or apparatus faults.
""".strip()


def create_zero_shot_prompt(
    experiment_context=None,
    analysis_task=generic_analysis_task,
    show_prompt=False,
):
    """Combine optional experiment context and an analysis task."""
    context = (experiment_context or "").strip()
    prompt_sections = []

    if context:
        prompt_sections.append(f"EXPERIMENT CONTEXT\n{context}")

    prompt_sections.append(f"ANALYSIS TASK\n{analysis_task.strip()}")
    full_prompt = "\n\n".join(prompt_sections)

    if show_prompt:
        print(f"------ FULL PROMPT ------\n{full_prompt}\n-------------------------")

    return full_prompt

---
## 4. Define one reusable Ising call

The function below fixes three common notebook problems:

* It validates and uses the paths passed to the function rather than a hidden global variable.
* It accepts one plot or multiple related plots.
* It lets you omit `experiment_context` to test how much the context changes the response.
* It returns the model's text so you can save or compare it, while also displaying readable Markdown by default.

Keep `temperature=0.2` and `max_tokens=8192` unless you have a specific reason to change NVIDIA's recommended zero-shot defaults.

In [ ]:
def ising_zero_shot(
    plots,
    experiment_context=None,
    analysis_task=generic_analysis_task,
    temperature=0.2,
    max_tokens=8192,
    show_prompt=False,
    display_answer=True,
):
    """Analyze calibration plots with optional experiment context."""
    plot_paths = resolve_plot_paths(plots)
    full_prompt = create_zero_shot_prompt(
        experiment_context=experiment_context,
        analysis_task=analysis_task,
        show_prompt=show_prompt,
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": full_prompt},
                    *[image_part(path) for path in plot_paths],
                ],
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    answer = response.choices[0].message.content or ""

    if display_answer:
        display(Markdown(answer))

    return answer

---
## 5. Example: Resonator Spectroscopy

We are now ready to start using NVIDIA Ising Calibration. Let's start with a resonator spectroscopy experiment. The generic analysis task defied above is a broad analysis. You can use this when you want Ising to describe the result, assess quality, extract supported parameters, and recommend what to do next. The example context supplied below for a resonator spectroscopy experiments shows the best practice method of providing experimental context.

Note: This first call sets `show_prompt=True` so you can inspect the complete prompt sent along with the image.

In [ ]:
resonator_context = """
Experiment type: Readout-resonator spectroscopy for a superconducting qubit.
Measurement objective: Estimate the resonator frequency and decide whether the
result is adequate to continue readout calibration.
X-axis: Probe frequency in GHz.
Y-axis: |S21| transmission magnitude in microvolts (µV).
Known settings: Probe power was held fixed during the frequency sweep.
Known limitations: A magnitude trace and fit are shown, but no phase trace,
fit residuals, goodness-of-fit statistic, or independent uncertainty estimate
is supplied.
""".strip()

resonator_answer = ising_zero_shot(
    plots="res_spec_v1.png",
    experiment_context=resonator_context,
    show_prompt=True,
)

---
## 6. Ask a focused follow-up question

A broad analysis is useful for orientation. A focused task is better when the next decision is already known. Here the requested **calibration diagnosis** is limited to resonator identification and readiness for the next readout step.

In [ ]:
resonator_diagnosis_task = """
Answer only these questions:

1. How many distinct resonator dips are visibly supported?
2. What center frequency is supported by the plot, with image-appropriate
   units and precision?
3. Is the raw signal adequate to continue readout setup?
4. What single follow-up measurement would most directly test that decision?

Separate visible observations from inferences. Assess the data separately from
the displayed fit, and return null for any unsupported numerical value.
""".strip()

focused_resonator_answer = ising_zero_shot(
    plots="res_spec_v1.png",
    experiment_context=resonator_context,
    analysis_task=resonator_diagnosis_task,
)

---
## 7. Analyze related plots together

Use multiple images when they are complementary views of the same experiment. For example, a Ramsey time trace and its frequency spectrum can jointly support a detuning estimate while exposing disagreement between the fit and the spectrum.

List the image roles explicitly in the context. Ising receives the images in the same order as the filenames.

In [ ]:
ramsey_context = """
Experiment type: Ramsey experiment on a superconducting qubit.
Measurement objective: Determine whether a single-frequency Ramsey model
supports reliable estimates of qubit-drive detuning and T2*.
Pulse sequence: Two pi/2 pulses separated by a variable evolution time.
Image 1: Population versus evolution time with a fitted oscillation and decay
envelope.
Image 2: FFT magnitude of the same data with the fitted frequency marked.
Known limitations: Only the plotted data and annotations are available; no
fit residuals or independent uncertainty estimates are supplied.
""".strip()

ramsey_answer = ising_zero_shot(
    plots=["ramsey_v1.png", "ramsey_v1_fft.png"],
    experiment_context=ramsey_context,
)

---
## 8. Reuse the zero-shot workflow

For each new calibration experiment:

1. Place the PNG or JPEG inside the `calibration_images` folder beside the notebook.
2. Optionally write context that names the experiment, objective, axes, units, settings, and known limitations.
3. Call `ising_zero_shot(...)` with the generic task for a broad first pass.
4. Replace `analysis_task` with a focused question when making a specific calibration decision.
5. Validate every extracted value and recommended action against the raw data and instructor guidance before changing hardware settings.

To retain an answer for later comparison, assign the return value to a variable as shown above. To suppress notebook rendering while still receiving the text, pass `display_answer=False`. 

You can also explore how the quality of your prompts impact the quality of Ising Calibration's response. Try removing context, providing inaccurate context, and asking for analysis results that can't be intreperted from the provided calibration plot. How does Ising Calibration perform?  Is this expected? 

Because context is optional, the smallest valid call is:

```python
no_context_answer = ising_zero_shot(plots="res_spec_v1.png")
```

Compare that answer with `resonator_answer`, which used the detailed `resonator_context`. Look for differences in interpretation, supported parameter extraction, uncertainty, and recommended next steps.

### When not to rely on Ising alone

Ising interprets the plot images and text you provide; it does not automatically inspect the raw measurement arrays, instrument logs, acquisition metadata, or physical device state. Do not use its response as the sole basis for:

* accepting numerical parameters when the plot omits readable axes, units, uncertainty, residuals, or fit-quality information;
* converting an uncalibrated readout signal into absolute qubit-state population;
* assigning a hardware or control-system root cause when several mechanisms could produce the same visible pattern;
* diagnosing drift, intermittency, or timing faults that require repeated measurements or time-ordered metadata; or
* making autonomous, safety-critical, or irreversible hardware changes.

> **Use Ising as an analysis assistant:** let it identify visible features, challenge a fit, and propose a bounded next test. Validate its conclusions against raw data, metadata, repeated measurements, instrument state, and domain-expert judgment before acting.


---
# Part II — In-Context Learning

---
## 9. Add one labeled reference without changing the query

**In-context learning (ICL)** gives the model solved examples inside the request before it analyzes a new case. A **labeled demonstration** contains one plot and an expert-reviewed analysis. The **query plot** is the new image the model must analyze.

This short exercise uses only two synthetic, three-panel Ramsey images:

* `ramsey_syn_v1.png` is the labeled reference. It contains apparent phase slips caused by a **phase-reference discontinuity** and a conventional Ramsey fit that should not be trusted for parameter extraction.
* `ramsey_syn_v2.png` is the new query. It shows a different realization of the same general failure pattern, but its diagnosis is not supplied to the model.

The query image, experimental context, and analysis task stay identical between the zero-shot and ICL calls. The only change is that the ICL request places the expert-reviewed reference before the query. ICL does not retrain Ising or change its model weights.

In [ ]:
synthetic_ramsey_context = """
Experiment type: A Ramsey experiment on a superconducting qubit.
Measurement objective: Assess whether the trace supports reliable Ramsey
detuning and coherence estimates and identify behavior that should be
investigated before updating calibration parameters.
Left panel: Normalized readout signal versus Ramsey evolution time in µs,
with a conventional single-frequency decaying-sinusoid fit and fitted values.
Middle panel: Data-minus-fit residuals versus Ramsey evolution time in µs.
Right panel: Normalized FFT magnitude of the same trace versus frequency in MHz,
with the fitted frequency marked.
Known limitations: No parameter uncertainty, acquisition segmentation metadata,
independent repeat, or alternative-model comparison is shown.
""".strip()

---
## 10. Define one reusable ICL call

The function below accepts any number of expert-reviewed demonstrations, but each demonstration and query uses exactly one image. This keeps the relationship between each image and its analysis unambiguous.

Each demonstration dictionary must contain:

* `plot`: a filename inside `calibration_images`
* `expert_analysis`: the reviewed answer paired with that image

It may also contain `experiment_context` with factual context for that reference measurement. Query context is optional as well.

The function sends one interleaved user message: reference prompt, reference image, expert analysis, and then the unchanged new-query prompt and image. Support examples come first and the query comes last, following the broad ICL structure used by QCalEval.

In [ ]:
def resolve_single_plot(plot):
    """Return one validated calibration-image path."""
    paths = resolve_plot_paths(plot)
    if len(paths) != 1:
        raise ValueError("Provide exactly one image for each reference and query.")
    return paths[0]


def ising_icl(
    demonstrations,
    query_plot,
    query_experiment_context=None,
    analysis_task=generic_analysis_task,
    temperature=0.2,
    max_tokens=8192,
    show_prompt_summary=False,
    display_answer=True,
):
    """Analyze one new plot after one or more expert-labeled examples."""
    if not demonstrations:
        raise ValueError("Provide at least one expert-labeled demonstration.")

    required_fields = {"plot", "expert_analysis"}
    content = [{
        "type": "text",
        "text": (
            "REFERENCE EXAMPLES\n"
            "Use the following expert-reviewed plot analyses as references "
            "for the same task applied to the new query. Match visible "
            "evidence rather than copying a diagnosis blindly."
        ),
    }]

    for index, example in enumerate(demonstrations, start=1):
        missing_fields = required_fields - set(example)
        if missing_fields:
            raise ValueError(
                f"Demonstration {index} is missing: "
                + ", ".join(sorted(missing_fields))
            )
        if not str(example["expert_analysis"]).strip():
            raise ValueError(f"Demonstration {index} has an empty analysis.")

        reference_prompt = create_zero_shot_prompt(
            experiment_context=example.get("experiment_context"),
            analysis_task=analysis_task,
        )
        reference_path = resolve_single_plot(example["plot"])
        content.extend([
            {
                "type": "text",
                "text": f"REFERENCE {index}\n{reference_prompt}",
            },
            image_part(reference_path),
            {
                "type": "text",
                "text": (
                    f"EXPERT ANALYSIS FOR REFERENCE {index}\n"
                    + str(example["expert_analysis"]).strip()
                ),
            },
        ])

    query_prompt = create_zero_shot_prompt(
        experiment_context=query_experiment_context,
        analysis_task=analysis_task,
    )
    query_path = resolve_single_plot(query_plot)
    content.extend([
        {"type": "text", "text": "NEW QUERY\n" + query_prompt},
        image_part(query_path),
    ])

    if show_prompt_summary:
        print(
            f"Prepared {len(demonstrations)} labeled reference(s) "
            f"and one new query: {query_path.name}"
        )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": content}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    answer = response.choices[0].message.content or ""

    if display_answer:
        display(Markdown(answer))

    return answer

---
## 11. Define the expert reference and evaluation criteria

The expert analysis below teaches the model how to connect visible phase discontinuities, structured residuals, and a multi-component FFT to an unreliable stationary-frequency fit. It also models appropriate caution: the plot supports an acquisition-block hypothesis, not a confirmed hardware cause.

Before running either condition, decide what improvement would mean for `ramsey_syn_v2.png`. A stronger answer could:

1. Notice repeated segment-dependent changes in oscillation phase.
2. Connect the fit's phase mismatch to structured residuals, multiple FFT components, and the displayed low $R^2=0.587$.
3. Reject the displayed $T_2^*=12.3\ \mu\mathrm{s}$ as a reliable calibration value.
4. Present acquisition-block phase-reference changes as a hypothesis rather than a confirmed apparatus fault.
5. Recommend checking whether discontinuity locations align with acquisition boundaries or reference-oscillator resets.

can you think of any others?

Remember: a longer answer is not automatically a better answer.

In [ ]:
phase_discontinuity_reference_analysis = """
1. TECHNICAL DESCRIPTION
The measured trace contains a decaying Ramsey oscillation with several abrupt
changes in phase at roughly regular positions. The red stationary-frequency
fit loses phase agreement in contiguous sweep regions. The residuals are large
and structured, and the FFT contains several nearby components.

2. EXPERIMENTAL CONCLUSION
One phase reference does not describe the complete sweep. The conventional fit
reports a frequency near the dominant FFT peak but does not adequately model
the data across the apparent segment boundaries.

3. EXPERIMENTAL SIGNIFICANCE
The fit can absorb phase inconsistency as faster decay, biasing T2* and possibly
detuning. Regular spacing suggests acquisition-block-dependent phase-reference
offsets, but the plot alone cannot establish the control or hardware cause.

4. FIT QUALITY ASSESSMENT
The displayed fit is unreliable for calibration extraction: R² is 0.602, the
red curve repeatedly misses the measured phase, and the residuals contain
coherent oscillatory structure far above the point-to-point noise level.

5. PARAMETER EXTRACTION
Displayed fit frequency: 0.619 MHz. Displayed fit T2*: 10.6 µs. These are fit
outputs, but reliable detuning and reliable T2* are null because the model is
visibly inadequate.

6. CALIBRATION STATUS CLASSIFICATION
ANOMALOUS_BEHAVIOR

7. RECOMMENDED NEXT STEP
Repeat the sweep while recording acquisition-block boundaries and reference-
oscillator phase-reset events, then test whether the phase-offset locations
align with those boundaries.
""".strip()


phase_discontinuity_demonstrations = [{
    "plot": "ramsey_syn_v1.png",
    "experiment_context": synthetic_ramsey_context,
    "expert_analysis": phase_discontinuity_reference_analysis,
}]

---
## 12. Run the zero-shot baseline

First, Ising receives only the standard prompt and `ramsey_syn_v2.png`. The model can use its pretrained calibration knowledge, but it does not see the labeled synthetic reference.

In [ ]:
synthetic_zero_shot_answer = ising_zero_shot(
    plots="ramsey_syn_v2.png",
    experiment_context=synthetic_ramsey_context,
    show_prompt=True,
)

---
## 13. Analyze the same query with ICL

Now Ising receives `ramsey_syn_v1.png` and its expert analysis before it receives the unchanged prompt and `ramsey_syn_v2.png`.

In [ ]:
synthetic_icl_answer = ising_icl(
    demonstrations=phase_discontinuity_demonstrations,
    query_plot="ramsey_syn_v2.png",
    query_experiment_context=synthetic_ramsey_context,
    show_prompt_summary=True,
)

---
## 14. Interpret and reuse the result

Compare the two displayed answers against the five criteria established above. ICL helped only if the second answer is more accurate, evidence-grounded, and appropriately cautious—not merely more similar to the reference wording. This demonstration does not prove that phase-reference discontinuities were absent from Ising's training data; it tests whether one relevant expert reference improves this particular analysis.

To reuse the function with your own data:

1. Put each reference image and the new query image in `calibration_images`.
2. Create one dictionary per reference with `plot` and an expert-reviewed `expert_analysis`; add `experiment_context` when it is useful.
3. Keep the same `analysis_task` for the references and query.
4. Call `ising_icl(demonstrations=..., query_plot=...)`, optionally adding `query_experiment_context=...`.
5. Keep the new query out of the demonstration list, and verify all model conclusions before changing device settings.

The phase-discontinuity example above is already a complete reusable template: replace the filename, context, and expert analysis in `phase_discontinuity_demonstrations`, then call `ising_icl` with a new query image.

## Conclusion

Remember, Ising is an analysis assistant, not an autonomous source of truth. Validate diagnoses, extracted parameters, and proposed actions against raw measurements, metadata, fit residuals, instrument state, and domain-expert judgment.

**Next steps:** Reuse zero-shot analysis for new experiment types, then add ICL only when you have relevant, expert-reviewed examples with a consistent task and answer format. For further context, see [NVIDIA Ising Calibration](https://build.nvidia.com/nvidia/ising-calibration-1.5-31b) and the [QCalEval paper](https://arxiv.org/abs/2604.25884).

**Related Notebooks:**
* [Quick Start to Quantum Computing](https://github.com/NVIDIA/cuda-q-academic/blob/main/quick-start-to-quantum/01_quick_start_to_quantum.ipynb) — reviews foundational qubits, gates, and measurements
* [QEC Noisy Simulation](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/03_QEC_Noisy_Simulation.ipynb) — explores noisy quantum behavior in a complementary workflow